# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import config

In [ ]:
C = config["context_len"]
h = config["horizon"]
vix_df = pd.read_csv('./data/VIXCLS.csv', parse_dates=['observation_date'], index_col='observation_date')
vix_df.dropna(inplace=True)
index = pd.DataFrame(index=vix_df.index)
index["row"] = np.arange(len(index))
dgs5_df = pd.read_csv('./data/DGS5.csv', parse_dates=['observation_date'], index_col='observation_date')
dgs5_df.dropna(inplace=True)
vix_df.head()

# Chronos Effective Token Range
Base on VIX in the range of 10 to 100, the plot shows the tokens in Chronos's vocabulary that occupy this range

In [ ]:
# rolling window mean
scaled_bin_centers = np.linspace(-15, 15, 4094)
vix_rolling_mean = vix_df.rolling(window=512).mean().dropna()
log_vix_rolling_mean = np.log(vix_df).rolling(window=512).mean().dropna()
raw_centers_vix = scaled_bin_centers * vix_rolling_mean.values
raw_centers_log_vix = np.exp(scaled_bin_centers * log_vix_rolling_mean.values)

# for each row find the indices where the value is between 10 and 100
indices_vix = []
indices_log_vix = []
for i in range(raw_centers_vix.shape[0]):
    indices = np.where((raw_centers_vix[i, :] > 10) & (raw_centers_vix[i, :] < 100))[0]
    indices_vix.append([indices.max()-indices.min(), indices.min()])
    indices = np.where((raw_centers_log_vix[i, :] > 10) & (raw_centers_log_vix[i, :] < 100))[0]
    indices_log_vix.append([indices.max()-indices.min(), indices.min()])
indices_vix = np.array(indices_vix)
indices_log_vix = np.array(indices_log_vix)

# plot the each rows min and max indices as a span
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].bar(np.arange(indices_vix.shape[0]), height=indices_vix[:, 0], bottom=indices_vix[:, 1], width=1, alpha=0.5, label='raw VIX')
ax[0].set_ylabel('Token')
ax[0].set_title('VIX Token Range Over Time')
ax[0].legend()
ax[1].bar(np.arange(indices_log_vix.shape[0]), height=indices_log_vix[:, 0], bottom=indices_log_vix[:, 1], width=1, alpha=0.5, label='log VIX')
ax[1].set_ylabel('Token')
ax[1].set_title('log VIX Token Range Over Time')
ax[1].legend()
plt.show()

# Stationarity test
Unit root test on VIX and log(VIX) 

In [ ]:
# stationarity test for VIX
from statsmodels.tsa.stattools import adfuller
col =vix_df.columns[0]
for i, func in enumerate([lambda x: x, np.log]):
    result = adfuller(func(vix_df.loc[:'2017-12-28', col]))
    name = "log " if i == 1 else "diff " if i == 2 else ""
    print(f"ADF test for {name}{col}: Stat: {result[0]:.5f} / p-value: {result[1]:.5f}")

# Metrics

Evaluate models based on config (Note current config takes ~4-5hours on a M4 Mac Studio)

In [ ]:
from backtest import run_backtest, evaluate
raw_df, fan_charts = run_backtest(config)
summary, df_scored = evaluate(raw_df, config)

In [ ]:
df_scored.to_csv(f"{config['out_dir']}/vix_test_per_origin_results_{config['horizon']}.csv", index=False)
summary.to_csv(f"{config['out_dir']}/vix_test_summary_metrics_{config['horizon']}.csv", index=True)
pd.set_option("display.width", 160)
print("\n================ SUMMARY (step " + str(config['horizon']) + ") ================")
print(summary)

# Regimes
Plot the forecast mean, median and intervals on specific regimes

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
vix_df["2018-01-01":].plot(ax=ax, legend=False)
colors = ["red", "orange", "green"]
for i, regime in enumerate(config['regimes']):
    start = config['regimes'][regime][0]
    end = config['regimes'][regime][1]
    start_plot = start
    start_h = vix_df.iloc[vix_df.index.get_loc(start) + config['horizon']].name
    end_h = vix_df.iloc[vix_df.index.get_loc(end) + config['horizon']].name
    plt.axvspan(start_h, end_h, color=colors[i], alpha=0.3, label=regime)
plt.title("VIX Regimes")
plt.xlabel("Date")
plt.legend()

In [ ]:
regimes={
    "pre-covid": ("2018-02-07", "2020-03-31"),
    "mid-covid": ("2018-02-24", "2020-05-30"),
    "peak-covid": ("2018-03-04", "2020-05-30"),
    "post-covid": ("2018-04-01", "2020-12-31"),
    # "pre-tariffs": ("2023-04-09", "2025-12-31"),
    # "mid-tariffs": ("2023-04-12", "2025-12-31"),
    # "peak-tariffs": ("2023-04-15", "2025-12-31"),
    # "pre-volmageddon": ("2016-01-21", "2019-04-10"),
    # "mid-volmageddon": ("2016-01-23", "2019-04-10"),
    # "peak-volmageddon": ("2016-01-24", "2019-04-10")
}
fig, ax = plt.subplots(len(regimes), 1, figsize=(20, 6*len(regimes)))
for i, regime in enumerate(regimes.keys()):
    cax = ax[i] if len(regimes) > 1 else ax
    cax.set_title(f"End date of context (origin): {vix_df[regimes[regime][0]:regimes[regime][1]].iloc[C-50:C].index[-1].strftime('%Y-%m-%d')}, Start date of forecast: {vix_df[regimes[regime][0]:regimes[regime][1]].iloc[C:C+h].index[0].strftime('%Y-%m-%d')}")
    vix_df[regimes[regime][0]:regimes[regime][1]].iloc[C-50:C].plot(ax=cax)
    vix_df[regimes[regime][0]:regimes[regime][1]].iloc[C:C+h].plot(ax=cax)
plt.tight_layout()

In [ ]:
from backtest import plot_regimes
model_names = ["aregarch", "chronos-raw"]
# model_names = ["aregarch", "chronos"]
# model_names = ["ar1", "chronos"]
# model_names = ["har", "aregarch", "ar1","chronos"]


plot_regimes(config, model_names, regimes)

# Residuals
Anlayse the residuals in the evaluation period

In [ ]:
from backtest import compute_residuals
residuals_df = compute_residuals(config, models=["har", "aregarch", "chronos"], step=1)
residuals_df.describe()

In [ ]:
from backtest import plot_residuals
plot_residuals(residuals_df, config)